In [140]:
# Packages
import os
import re

# For downloading online NOAA data
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor
import requests

# For data analysis and visualization
import pandas as pd
import numpy as np
import zipfile
import matplotlib.pyplot as plt
import geopandas as gpd

# pd.set_option('display.max_colwidth', None)

In [151]:
# Links of gzip storm data from NOAA website: https://www.ncei.noaa.gov/stormevents/ftp.jsp

def storm_data():
    # Web scrape NOAA weather data links
    storms_url = 'https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/'
    req = requests.get(storms_url)                  # access url webpage
    soup = BeautifulSoup(req.text, 'html.parser')   # parse thru HTML text of webpage

    # Find 2000s data.csv.gz filenames in <a href="link" > format 
    pattern = r'StormEvents_details-ftp_v1\.0_d20\d{2}_c\d{8}\.csv\.gz'   # 2000s filename pattern
    data_links = []
    for link in soup.find_all('a', attrs={'href': re.compile(pattern)}):
        year_data = link.get('href')
        full_link = str(storms_url) + str(year_data)
        data_links.append(full_link)
    return data_links


# Parallel download func for data
def download_files(data):
    # Create new directory for data
    wx_data_dir = '../data/noaa'
    os.makedirs(wx_data_dir, exist_ok=True)
    
    # Check url request for 'content-disposition' header to parse .gz filenames
    response = requests.get(data, stream=True)
    if 'content-disposition' in response.headers:
        content_disp = response.headers['content-disposition']
        file_name = content_disp.split('filename=')[1]
    else:
        file_name = data.split('/')[-1]
    
    # Write downloaded gzip data to data dir
    gz_name = os.path.join(wx_data_dir, file_name)
    with open(gz_name, 'wb') as gz_file:
        gz_file.write(response.content)
    # print(f'Downloaded file to {gz_name}')


# Use ThreadPoolExecutor() to parallel download gzip files
with ThreadPoolExecutor() as executor:
    executor.map(download_files, storm_data())

In [235]:
# Create pandas dataframes of data for each year (optional: state)
# Info about files: https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/Storm-Data-Bulk-csv-Format.pdf 
def weather_df(year, state=None):
    # Look for selected year's data from data/noaa dir
    file_pattern = rf'StormEvents_details-ftp_v1\.0_d{year}_c\d{{8}}\.csv\.gz'
    gz_files = os.listdir('../data/noaa')
    try:
        match = [g for g in gz_files if re.search(file_pattern, g)][0]
    except IndexError:
        print('No matches found! Did you select a year between 2000 and 2026?')
    
    # Adjust date columns to YYYMMDD format
    df = pd.read_csv(f'../data/noaa/{match}', compression='gzip', header=0)
    df['BEGIN_DAY'] = df['BEGIN_DAY'].apply(lambda x: '0'+str(x) if len(str(x))<2 else str(x))
    df['BEGIN_DATE'] = df['BEGIN_YEARMONTH'].astype(str) + df['BEGIN_DAY']
    df['END_DAY'] = df['END_DAY'].apply(lambda x: '0'+str(x) if len(str(x))<2 else str(x))
    df['END_DATE'] = df['END_YEARMONTH'].astype(str) + df['END_DAY']
    
    # Adjust for details & events columns
    ## EVENT_IDs are unique for each entry
    ## MAGNITUDE: wind speeds (knots), hail (inches)
    details = ['EVENT_ID', 'BEGIN_DATE', 'END_DATE', 'STATE', 'STATE_FIPS', 
               'CZ_NAME', 'EVENT_TYPE', 'INJURIES_DIRECT', 'INJURIES_INDIRECT', 
               'DEATHS_DIRECT', 'DEATHS_INDIRECT', 'DAMAGE_PROPERTY', 
               'MAGNITUDE', 'TOR_F_SCALE', 'EPISODE_NARRATIVE']
    event_types = ['Blizzard', 'Cold/Wind Chill', 'Drought', 'Excessive Heat',
                   'Extreme Cold/Wind Chill', 'Flash Flood', 'Flood', 'Hail', 
                   'Heat', 'Heavy Rain', 'Heavy Snow', 'Hurricane (Typhoon)', 
                   'Ice Storm', 'Sleet', 'Storm Surge/Tide', 'Thunderstorm Wind', 
                   'Tornado', 'Tropical Storm', 'Tsunami', 'Wildfire', 
                   'Winter Storm', 'Winter Weather']
    df = df[details]
    df = df[df['EVENT_TYPE'].isin(event_types)]
    
    # Make rows for each date that events continued through 
    df['days'] = df['END_DATE'].astype(int) - df['BEGIN_DATE'].astype(int)
    rdf = pd.DataFrame(np.repeat(df.values, repeats=df['days']+1, axis=0), columns=df.columns)
    rdf['repeat'] = 1
    rdf['repeat'] = rdf.groupby('EVENT_ID')['repeat'].cumsum() - 1
    rdf['BEGIN_DATE'] = rdf['BEGIN_DATE'].astype(int) + rdf['repeat']
    rdf = rdf.drop(columns=['END_DATE', 'repeat', 'days'])
    rdf = rdf.rename(columns={'BEGIN_DATE': 'DATE'})
    wx_df = rdf
    
    # Remove non-continental states, territories
    wx_df = wx_df.loc[~wx_df['STATE'].isin(['ALASKA', 'HAWAII', 'GUAM', 
                                            'AMERICAN SAMOA', 'PUERTO RICO', 
                                            'VIRGIN ISLANDS'])]
    
    # Filter by state if desired
    if state is not None:
        state = state.upper()
        wx_df = wx_df[wx_df['STATE'] == state]
    
    wx_df = wx_df.sort_values(by=['DATE'], ascending=True).reset_index(drop=True)
    wx_df = wx_df.astype(str)
    return wx_df

In [236]:
us_2025 = weather_df(2025)
# us_2025.head(5)
us_2025['STATE'].unique()

<StringArray>
[            'ARKANSAS',                'TEXAS',          'CONNECTICUT',
            'TENNESSEE',           'NEW MEXICO',             'NEBRASKA',
             'COLORADO',                'IDAHO',             'MARYLAND',
              'VERMONT',             'NEW YORK',                'MAINE',
            'MINNESOTA',               'OREGON',         'NORTH DAKOTA',
                 'IOWA',              'ALABAMA',         'SOUTH DAKOTA',
              'WYOMING',           'CALIFORNIA',              'MONTANA',
         'PENNSYLVANIA',             'VIRGINIA',        'WEST VIRGINIA',
             'MICHIGAN',       'NORTH CAROLINA',                 'OHIO',
               'NEVADA', 'DISTRICT OF COLUMBIA',           'WASHINGTON',
               'KANSAS',             'MISSOURI',          'MISSISSIPPI',
             'ILLINOIS',              'INDIANA',            'WISCONSIN',
             'KENTUCKY',            'LOUISIANA',             'OKLAHOMA',
           'NEW JERSEY',             

In [201]:
# Unzip state & county geodata
# Boundary data downloaded from https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html 
state_dir = os.makedirs('../data/geodata/state', exist_ok=True)
with zipfile.ZipFile('../data/geodata/tl_2025_us_state.zip', 'r') as states:
    states.extractall('../data/geodata/state')
county_dir = os.makedirs('../data/geodata/county', exist_ok=True)
with zipfile.ZipFile('../data/geodata/tl_2025_us_county.zip', 'r') as counties:
    counties.extractall('../data/geodata/county')

In [ ]:
# Load df of county geographic boundaries
ctgeo_df = gpd.read_file('../data/geodata/county/tl_2025_us_county.shp')
ctgeo_df[ctgeo_df['STATEFP']=='01'].iloc[:,:-1]
# just want statefp, name, geometry
# stgeo_df = gpd.read_file('../data/geodata/state/tl_2025_us_state.shp')
# stgeo_df['NAME'].unique()

,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON
15,01,075,00161563,01075,0500000US01075,Lamar,Lamar County,06,H1,G4020,NaN,NaN,NaN,A,1566381027,1600679,+33.7870852,-088.0874309
39,01,025,00161538,01025,0500000US01025,Clarke,Clarke County,06,H1,G4020,NaN,NaN,NaN,A,3207494085,36657889,+31.6855211,-087.8186244
48,01,123,00161587,01123,0500000US01123,Tallapoosa,Tallapoosa County,06,H1,G4020,194,10760,NaN,A,1855828177,128760802,+32.8633076,-085.7996176
127,01,079,00161565,01079,0500000US01079,Lawrence,Lawrence County,06,H1,G4020,290,19460,NaN,A,1788864897,68681744,+34.5297760,-087.3218651
194,01,059,00161555,01059,0500000US01059,Franklin,Franklin County,06,H1,G4020,250,40770,NaN,A,1641945980,32539347,+34.4419892,-087.8428144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3012,01,057,00161554,01057,0500000US01057,Fayette,Fayette County,06,H1,G4020,NaN,NaN,NaN,A,1625693253,4330881,+33.7161568,-087.7642923
3013,01,017,00161534,01017,0500000US01017,Chambers,Chambers County,06,H1,G4020,122,29300,NaN,A,1545081616,16975680,+32.9155039,-085.3940321
3134,01,047,00161549,01047,0500000US01047,Dallas,Dallas County,06,H1,G4020,388,42820,NaN,A,2534926511,39117865,+32.3335263,-087.1143600
3151,01,125,00161588,01125,0500000US01125,Tuscaloosa,Tuscaloosa County,06,H1,G4020,NaN,46220,NaN,A,3421019929,78663574,+33.2902197,-087.5227834


In [ ]:
# Merge county df with wx_df based on state, state_fips, cz_name

,EVENT_ID,DATE,STATE,STATE_FIPS,CZ_NAME,EVENT_TYPE,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,MAGNITUDE,TOR_F_SCALE,EPISODE_NARRATIVE
704,1223153,20250103,DISTRICT OF COLUMBIA,11,DISTRICT OF COLUMBIA,Winter Weather,0,0,0,0,0.00K,NaN,NaN,An area of low pressure moved off into New England bringing a cold front through Virginia. Snow showers and squalls along the front brought snow accumulations of a coating to an inch.
1956,1222904,20250105,DISTRICT OF COLUMBIA,11,DISTRICT OF COLUMBIA,Winter Storm,0,0,0,0,0.00K,NaN,NaN,An area of low pressure tracked across southern Virginia bringing the first widespread accumulating snow of the season. Snow overspread the area during the late evening hours of January 5th into the overnight. Snow overspread the area during the late evening hours of January 5th into the overnight. This continued steady through mid-morning. A lull occurred during the afternoon with a few snow showers. Another round of snow on the back side of the upper level low moved in during the evening bringing additional accumulations. This resulted in of 7 to 10 inches of snow across the District of Columbia. Winds gusted to 45 mph resulting in blowing snow at the end of the storm.
2255,1222904,20250106,DISTRICT OF COLUMBIA,11,DISTRICT OF COLUMBIA,Winter Storm,0,0,0,0,0.00K,NaN,NaN,An area of low pressure tracked across southern Virginia bringing the first widespread accumulating snow of the season. Snow overspread the area during the late evening hours of January 5th into the overnight. Snow overspread the area during the late evening hours of January 5th into the overnight. This continued steady through mid-morning. A lull occurred during the afternoon with a few snow showers. Another round of snow on the back side of the upper level low moved in during the evening bringing additional accumulations. This resulted in of 7 to 10 inches of snow across the District of Columbia. Winds gusted to 45 mph resulting in blowing snow at the end of the storm.
4859,1223107,20250110,DISTRICT OF COLUMBIA,11,DISTRICT OF COLUMBIA,Winter Weather,0,0,0,0,NaN,NaN,NaN,An area of low pressure tracked south of the region across the Carolinas. This brought a period of light to moderate snow to the District of Columbia during the late evening hours of January 10th through the predawn of January 11th. Snowfall accumulations ranged 1 to 2 inches.
5569,1223107,20250111,DISTRICT OF COLUMBIA,11,DISTRICT OF COLUMBIA,Winter Weather,0,0,0,0,NaN,NaN,NaN,An area of low pressure tracked south of the region across the Carolinas. This brought a period of light to moderate snow to the District of Columbia during the late evening hours of January 10th through the predawn of January 11th. Snowfall accumulations ranged 1 to 2 inches.
8538,1224837,20250120,DISTRICT OF COLUMBIA,11,DISTRICT OF COLUMBIA,Cold/Wind Chill,0,0,0,0,0.00K,NaN,NaN,Arctic air moved in behind a departing low pressure system on January 20. This Arctic air remained overhead as high pressure dominated through January 22. Winds gusted to 25-35 mph at times bringing apparent temperatures/wind chill values as low as -10F across the District of Columbia. Air temperatures fell into the single digits at night.
10387,1224837,20250121,DISTRICT OF COLUMBIA,11,DISTRICT OF COLUMBIA,Cold/Wind Chill,0,0,0,0,0.00K,NaN,NaN,Arctic air moved in behind a departing low pressure system on January 20. This Arctic air remained overhead as high pressure dominated through January 22. Winds gusted to 25-35 mph at times bringing apparent temperatures/wind chill values as low as -10F across the District of Columbia. Air temperatures fell into the single digits at night.
11352,1224837,20250122,DISTRICT OF COLUMBIA,11,DISTRICT OF COLUMBIA,Cold/Wind Chill,0,0,0,0,0.00K,NaN,NaN,Arctic air moved in behind a departing low pressure system on January 20. This Arctic air remained overhead as high pressure dominated through January 22. Winds gusted to 25-35 mph at times bringing apparent temperatu